# Chebyshev Policies and the Mountain Car Problem: Reinforcement Learning for Low-dimensional Control Tasks
## REINFORCE utilizing Chebyshev polynomial basis  

Experiments for ICML 2026 paper "Chebyshev Policies and the Mountain Car Problem: Reinforcement Learning for Low-dimensional Control Tasks".  
In this notebook, we train Chebyshev policies on the Mountain Car problem using REINFORCE.    

https://pytorch.org/docs/stable/distributions.html:  
_REINFORCE is commonly seen as the basis for policy gradient methods in reinforcement learning_  

Version 3.0  
Date: 2026-01-29  
Current version: hannes.unger@fh-salzburg.ac.at, stefan.huber@fh-salzburg.ac.at     

In [ ]:
import torch
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import multiprocessing as mp
from itertools import repeat
from utils import parallel
from pickleshare import PickleShareDB
from matplotlib import colors as mcolors
from algorithms import multivariate_polynomial_basis

db = PickleShareDB('./picklesharedb')

%load_ext autoreload
%autoreload 2

In [ ]:
def get_best_candidates_of_training_result(results, optimizer_list, n_runs):
    best_index = {}
    for i in range(len(optimizer_list)):
        valid_results = []
        for j in range(n_runs):
            # Check if the element is a list and has the structure we're looking for
            if isinstance(results[i][j], list) and len(results[i][j]) > 1:
                valid_results.append((j, results[i][j][1][-1])) # take only last reward
        
        if valid_results:  # Only proceed if there are valid results
            best_index[i] = max(valid_results, key=lambda x: x[1])[0]
        else:
            print(f'Finding index: No converging result for optimizer {i}')
            # Optionally skip optimizers with no valid results

    best_coeffs = []
    to_delete = []

    for index in best_index:
        try:
            best_coeffs.append(results[index][best_index[index]][-1])
        except:
            print(f'Adding coeffs: No converging result for optimizer {index}, removing entry')
            to_delete.append(index)
    
    for i in to_delete:
        del best_index[i]
    
    print(f'{best_index}\n{len(best_index)}\n{len(best_coeffs)}')

    return best_index, best_coeffs

In [ ]:
def get_converging_coeffs_of_training_results(results, optimizer_list, n_runs):
    coeffs = []

    for i, opt_result in enumerate(results):
        coeffs.append([])
        for j in range(n_runs):
            try:
                coeffs[i].append(opt_result[j][-1])
            except:
                print(f'Adding coeffs: No converging result for optimizer {i}')

    to_delete = []

    for i in range(0, len(coeffs)):
        if len(coeffs[i]) == 0:
            to_delete.append(i)

    coeffs = [val for i, val in enumerate(coeffs) if i not in to_delete]
    optimizer_list = [val for i, val in enumerate(optimizer_list) if i not in to_delete]
    return coeffs, optimizer_list

In [ ]:
def plot_evaluation_rewards(results, optimizers, window_size=5, axes=None):
    reinforce_results_eval_optimizers_mean_rewards = []
    for i in range(len(results)):
        reinforce_results_eval_optimizers_mean_rewards.append(np.mean(np.array([results[i][j] for j in range(len(results[i]))]), axis=0)) 

    # moving average for n_runs for optimizer
    window_size=5
    reinforce_results_eval_optimizers_moving_average_rewards = []

    for i in range(len(results)):
        reinforce_results_eval_optimizers_moving_average_rewards.append(np.convolve(reinforce_results_eval_optimizers_mean_rewards[i], np.ones(window_size), mode='valid') / window_size)

    if axes is None:
        fig, axes = plt.subplots(4, 3) 
        axes = axes.flatten()
        fig.set_figwidth(len(optimizers)*3)
        fig.set_figheight(20)
        fig.suptitle(f'Rewards over episodes with different optimizers')

    for i, opt in enumerate(optimizers):
        ax = axes[i]
        ax.set_title(f'{opt}')
        for j in range(len(results[i])):   
            ax.plot(results[i][j], alpha=0.3)
            ax.set_ylim([-100, 100])
        ax.plot(reinforce_results_eval_optimizers_moving_average_rewards[i], 'k', label='moving average')

In [ ]:
def plot_evaluation_min_mean_max_rewards(results, optimizers, window_size=5, color='r', axes=None):
    # mean rewards
    reinforce_results_eval_optimizers_min_rewards = []
    reinforce_results_eval_optimizers_mean_rewards = []
    reinforce_results_eval_optimizers_max_rewards = []
    for i in range(len(results)):
        reinforce_results_eval_optimizers_mean_rewards.append(np.mean(np.array([results[i][j][1] for j in range(len(results[i]))]), axis=0)) 
        reinforce_results_eval_optimizers_min_rewards.append(np.min(np.array([results[i][j][1] for j in range(len(results[i]))]), axis=0)) 
        reinforce_results_eval_optimizers_max_rewards.append(np.max(np.array([results[i][j][1] for j in range(len(results[i]))]), axis=0)) 

    # moving average for n_runs for optimizer
    reinforce_results_eval_optimizers_moving_average_rewards = []

    for i in range(len(results)):
        reinforce_results_eval_optimizers_moving_average_rewards.append(np.convolve(reinforce_results_eval_optimizers_mean_rewards[i], np.ones(window_size), mode='valid') / window_size)

    if axes is None:
        fig, axes = plt.subplots(4, (len(optimizers)+2)//4) 
        axes = axes.flatten()
        fig.set_figwidth(len(optimizers)*3)
        fig.set_figheight(20)
        fig.suptitle(f'Rewards over episodes with different optimizers')

    for i, opt in enumerate(optimizers):
        ax = axes[i]
        ax.set_title(f'{opt}')        
        ax.plot(reinforce_results_eval_optimizers_max_rewards[i], color, alpha=0.5)
        ax.plot(reinforce_results_eval_optimizers_min_rewards[i], color, alpha=0.5)
        ax.plot(reinforce_results_eval_optimizers_mean_rewards[i], color)
        ax.fill_between(x=range(len(reinforce_results_eval_optimizers_max_rewards[i])), y1=reinforce_results_eval_optimizers_min_rewards[i], y2=reinforce_results_eval_optimizers_max_rewards[i], color=color, alpha=0.1)
        ax.set_ylim([-100, 100])

In [ ]:
def plot_training_min_mean_max_rewards(results, all_optimizers, color='r', axes=None):
    n_runs = len(results[0])
    episodes = len(results[0][0][1])

    results_mean_rewards = []
    results_min_rewards = []
    results_max_rewards = []
    for i in range(len(all_optimizers)):
        try:
            results_mean_rewards.append(np.mean(np.array([results[i][j][1] for j in range(n_runs) if isinstance(results[i][j], list)]), axis=0)) # skip rows where nan values lead to exception instead of list
        except:
            results_mean_rewards.append(np.nan)
        try:
            results_min_rewards.append(np.min(np.array([results[i][j][1] for j in range(n_runs) if isinstance(results[i][j], list)]), axis=0)) 
        except:
            results_min_rewards.append(np.nan)
        try:    
            results_max_rewards.append(np.max(np.array([results[i][j][1] for j in range(n_runs) if isinstance(results[i][j], list)]), axis=0)) 
        except:
            results_max_rewards.append(np.nan)


    if axes is None:
        fig, axes = plt.subplots(5, (len(all_optimizers)+2)//4) 
        axes = axes.flatten()
        fig.set_figwidth(len(all_optimizers)*3)
        fig.set_figheight(20)
        fig.suptitle(f'Rewards over episodes with different optimizers, {episodes} episodes, {n_runs} runs')

    for i, _ in enumerate(all_optimizers):
        try:
            ax = axes[i]
            ax.set_title("%s" % all_optimizers[i])
            ax.plot(results_min_rewards[i], color, alpha=0.5)
            ax.plot(results_max_rewards[i], color, alpha=0.5)
            ax.plot(results_mean_rewards[i], color)
            ax.fill_between(x=range(len(results_min_rewards[i])), y1=results_min_rewards[i], y2=results_max_rewards[i], color=color, alpha=0.1)
            ax.set_ylim([-100, 100])
        except:
            pass

In [ ]:
def get_best_candidates_of_evaluation_result(evaluation_results):
    max_mean = float('-inf')
    outer_index = -1
    inner_index = -1

    for i, outer in enumerate(evaluation_results):
        for j, inner in enumerate(outer):
            #mean_value = sum(inner) / len(inner)  # Compute mean of current inner list
            mean_value = np.mean(inner[1])
            if mean_value > max_mean:
                max_mean = mean_value
                outer_index = i
                inner_index = j

    print(f"The optimizer index with the highest mean is: {outer_index}")
    print(f"The corresponding policy index is: {inner_index}")

    return outer_index, inner_index

In [ ]:
def get_best_single_episode_evaluation_result(evaluation_results, mrp):
    max_reward = float('-inf')
    optimizer = -1
    run = -1

    for i, outer in enumerate(evaluation_results):
        for j, inner in enumerate(outer):
            if inner[0] > max_reward:
                max_reward = inner[0]
                optimizer = i
                run = j

    print(f"The optimizer index with the highest reward ({max_reward}) is: {optimizer}")
    print(f"The corresponding policy index is: {run}")
    s = len(evaluation_results[optimizer][run][1])-2
    print(f"Episode length: {s}")
    v = mrp.unnormalize(evaluation_results[optimizer][run][1][-1], np.array([0.6, 0.07]), np.array([-1.2, -0.07]))[1]
    print(f"Target velocity: {v}")

    return max_reward, optimizer, run, s, v

### Play around with environment

In [ ]:
env = gym.make("MountainCarContinuous-v0", render_mode="human") 
env.reset() # Resets the environment to the initial state and returns that state

In [ ]:
# Continuous action (applied force) in the range [-1, 1], with negative values pushing left and positive ones right
env.action_space

In [ ]:
# Random action
env.action_space.sample()

In [ ]:
# ?
env.observation_space

In [ ]:
# Random observation, gives state [pos, vel]
env.observation_space.sample()

In [ ]:
# Random action
env.action_space.sample()

In [ ]:
# Random step with random action, returns (next_state, reward, terminated, truncated, info)
env.step(env.action_space.sample())

In [ ]:
# # Run some random actions
# for i in range(200):
#     env.render()
#     obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
#     if terminated:
#         s = env.reset()

# Polynomial Approximator

### Max-Degree and Power Basis

We are interested in approximating functions $\mathbb{R}^n \to \mathbb{R}$, or tabulations thereof, by multi-variate polynomials.  
Let $(x_1, \dots, x_n) \in \mathbb{R}^n$ then we call $f = \prod_{i=0}^n x_i^{d_i}$ a monomial and its degree $\deg f$ is $\sum_i d_i$.  
A linear combinations of monomials we call a polynomial and its degree is the maximum degree over its monomials.  
  
For our purposes it will be convenient to introduce another notion of degree refering to the largest exponent of its exponent:  
We define the _max-degree_ $\deg^* f$ of a monomial $f$ as $\max_i d_i$ and the max-degree of a polynomial as the maximum max-degree of its monomials.  
For the uni-variate case, where $n=1$, the terms degree and max-degree are the same.  
The space of multi-variate polynomials of max-degree at most $d$ is a vector space $P^n_d$, i.e.,  
closed under addition and scalar-multiplication. The monomials $\prod_{i=0}^n x_i^{d_i}$ with $0 \le d_i \le d$ form a basis,  
and we call it the _power basis_. Consequently, the dimension of $P^n_d$ is $(d+1)^n$.  

In [ ]:
xss = np.linspace(-1, 1, 20)
yss = np.linspace(-1, 1, 20)
X, Y = np.meshgrid(xss, yss)

nrows, ncols = 4, 4
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, subplot_kw=dict(projection='3d'), figsize=(12,12))
fig.suptitle("Power basis")

for i in range(nrows):
    for j in range(ncols):

        p = multivariate_polynomial_basis.multivarpoly_by_univar(multivariate_polynomial_basis.univar_power, [i, j])

        zss = np.array([[p([x,y]) for x in xss] for y in yss])
        
        ax = axs[i][j]        
        ax.plot_surface(X, Y, zss, alpha=0.75, rstride=1, cstride=1, color='orangered', edgecolors='k', lw=0.6)
        ax.set_title(fr'$x_0^{i}·x_1^{j}$')

plt.subplots_adjust(hspace=0.5)
plt.show()

### Chebyshev Basis
  
Let $T_n \colon [-1, 1] \to \mathbb{R} \colon x \mapsto \cos(n \cdot \arccos(x))$ denote the $n$-th Chebyshev polynomial.  
It can be recursively defined as $T_{n+1}(x) = 2 x \cdot T_n(x) - T_{n-1}(x)$ together with $T_0(x) = 1$ and $T_1(x) = x$.  
The degree of $T_n$ is therefore $n$. Furthermore, we see by its definition that all its extrema are attained at absolute function values $1$.  
Also Chebyshev polynomials are orthogonal w.r.t. a weighted inner product 
$<f,g>_w = \int_{-1}^1 f(x) g(x) \; w(x) \mathrm{d}x$  
where the weight $w(x) = \frac{1}{\sqrt{1-x^2}}$. One can check that $<T_i, T_j>_w = \delta_{ij}$,  
where $\delta_{ij}$ denotes the Kronecker delta, i.e., the $T_i$ are orthogonal.  

We can generalize Chebyshev polynomials to $n$-variate Chebyshev polynomials $T_{d_1, \dots, d_n}(x_1, \dots, x_n) = \prod_i T_{d_i}(x_i)$.  
Note that the $\deg T_{d_1, \dots, d_n}$ is $\sum_i d_i$ and $\deg^* T_{d_1, \dots, d_n}$ is $\max_i d_i$.  
Also note that  
$ \int_{-1}^1 \cdots \int_{-1}^1 T_{d_1, \dots, d_n}(x_1, \dots, x_n) \cdot T_{u_1, \dots, u_n}(x_1, \dots, x_n) \; \prod_{i=1}^n w(x_i) \mathrm{d} x_1 \dots \mathrm{d} x_n =  \prod_{i=1}^n \int_{-1}^1 T_{d_i}(x_i) \cdot T_{u_i}(x_i) \; w(x_i) \mathrm{d} x_i = \prod_i \delta_{d_i u_i},$  
and hence we have orthogonality with respect to this $n$-dimensional generalization of the weighted inner product again.  
However, from that follows that there are $(d+1)^n$ linearly independent $n$-variate Chebyshev polynomials of max-degree at most $d$,  
and hence they form a basis of $P^n_d$ again. In particular, they are a good choice for function approximation.  

In [ ]:
xss = np.linspace(-1, 1, 20)
yss = np.linspace(-1, 1, 20)
X, Y = np.meshgrid(xss, yss)

nrows, ncols = 4, 4
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, subplot_kw=dict(projection='3d'), figsize=(12,12))
fig.suptitle("Chebyshev basis")

for i in range(nrows):
    for j in range(ncols):

        p = multivariate_polynomial_basis.multivarpoly_by_univar(multivariate_polynomial_basis.univar_chebychev, [i, j])

        zss = np.array([[p([x,y]) for x in xss] for y in yss])
        
        ax = axs[i][j]        
        ax.plot_surface(X, Y, zss, alpha=0.75, rstride=1, cstride=1, color='orangered', edgecolors='k', lw=0.6)
        ax.set_title(fr'$T_{i}(x_0)·T_{j}(x_1)$')

plt.subplots_adjust(hspace=0.5)
plt.show()

# What can we approximate?

## Power Basis

## State-Value function

In the mountain car example, we receive a state vector consisting of $m=2$ values with every observation:  
\
$s=\begin{pmatrix}x_0\\ x_1\end{pmatrix}$
\
We can represent this state by the feature vector of max-degree $d = 2$ using Power basis.  
Considering the monomials are $\prod_{i=0}^n x_i^{d_i}$ with $0 \le d_i \le d$ for $(x_1, \dots, x_n) \in \mathbb{R}^n$ as desribed above, we get the feature vector   

\
$\vec{x}(s)=\begin{pmatrix}x_0^0x_1^0\\ x_0^0x_1^1\\ \dots \\ x_0^2x_1^1\\ x_0^2x_1^2\end{pmatrix}$,

which consists of $(d+1)^{m}=9$ rows.  
Our value function then can be represented as the inner product of a weight vector $\vec{w}$ of the same dimension and this feature vector (S&B p.205):  
\
$\hat{v}(s) = \vec{w}^T \cdot \vec{x}(s) = \sum w_i x_i(s)$.  
\
This means that, for the prediction task, we are learning the coefficients $w$ of the *linear* combination  
\
$\hat{v}(s) = w_0 \cdot x_0^0x_1^0 + w_1 \cdot x_0^0x_1^1 + \dots + w_8 \cdot x_0^2x_1^2$.  

The approximator is linear in the weights to be learned (We could set up a linear equation system with sufficient points (=observations $x_s$) and solve for w.)    

## Action-value function (Q-Function)

For each input state *and action*, the Q-function gives the value of taking that action under that state.  
This means we have a further input variable, which can be represented by another dimension in our feature vector of max-degree $2$  
\
$\vec{x}(s, a)=\begin{pmatrix}x_0^0x_1^0a^0\\ x_0^0x_1^0a^1\\ \dots \\ x_0^2x_1^2a^1\\ x_0^2x_1^2a^2\end{pmatrix}$,  
 
Our approximated action-value function then again can be represented as the inner product of a weight vector $\vec{w}$ of the same dimension and this feature vector (S&B p.246):  
\
$\hat{q}(s,a) = \vec{w}^T \cdot \vec{x}(s,a) = \sum w_i x_i(s,a)$.  
  
A policy can then be derived by e.g. greedy maximization $\argmax_a \hat{q}(s,a)$.  
However, greedy maximization is costly: We have to find the global maximum over the  
entire action space. Better: adapting policy parameters proportionally to the gradient $\nabla \hat{q}$.  

## Policy

While $\vec{w}^T$ denotes weights/parameters of value functions, $\vec{\theta}$ denotes the weights/parameters of policies.  
Analogous to our value function, we can model our policy as a product of the parameter vector $\theta$ and the state feature vector:  
\
$\pi(s) = \vec{\theta}^T \cdot \vec{x}(s) = \sum \theta_i x_i(s)$.  


## Chebyshev Basis

What we have said also applies to Chebyshev basis, just that the our basis vectors are now defined by $\prod_i T_{d_i}(x_i)$.  
The state can then be represented by the feature vector   
\
$\vec{x}(s)=\begin{pmatrix}T_{0}(x_0)T_{0}(x_1)\\ T_{0}(x_0)T_{1}(x_1)\\ \dots \\ T_{2}(x_0)T_{1}(x_1)\\ T_{2}(x_0)T_{2}(x_1)\end{pmatrix}$,

which again consists of $(d+1)^{m}=9$ rows. Representations of state-value, action-value or policy functions can be formulated analogously to power basis.  

## Dynamics

We can also model the environment's dynamics using a polynomial model.  
It can then be used for e.g. Model-based RL or planning methods like Dynamic Programming with Policy iteration / Value iteration.  
In the case of this Mountain Car example, the dynamics are known and we could estimate the true value function using Dynamic Programming:  
\
$v_{t+1} = v_{t+1} + F \cdot P - 0.0025 * cos(3 * p_t)$ 
  
$p_{t+1} = p_t + v_{t+1}$  

However, at the moment, we don't want to go for model-based RL.

## Exploration vs. Exploitation - A stochastic policy

To enable exploration during training, we need to add a certain amount of random action selection to the process.  
For this, we let our policy evaluate to $\mu$ which we use to sample our action from a normal (Gaussian) distribution.  
A second approximator evaluates to $\sigma$, at the specific position $s$.  
So: $\mu$ sets the center of the distribution, $\sigma$ defines the range around which samples values vary.  
Following the notation of S&B, $\pi(a|s)$ is the probability of taking action $a$ in state $s$ under the stochastic policy $\pi$.  
**For evaluation outside training, we set $\sigma=0$ for a deterministic policy.**

In [ ]:
x = 0.25
sigma = 1e-12
torch.distributions.Normal(loc=x, scale=sigma).sample() # Deterministic policy: sample from distribution with center at x and variance almost 0. a variance of exactly 0.0 would return nan

In [ ]:
action = 0.25
torch.distributions.Normal(loc=x, scale=1e-12).cdf(torch.tensor(action)) # probability that normal random variable takes value up to including action

In [ ]:
# If the action matches the mean, the cdf is 0.5 (half of the values are left to the mean), if it exceeds the mean + sigma value, we get a probability of 1.0
action = 0.27
torch.distributions.Normal(loc=x, scale=1e-12).cdf(torch.tensor(action)) # probability that normal random variable takes value up to including action

In [ ]:
sigma = 0.5
torch.distributions.Normal(loc=x, scale=0.5).sample() # Stochastic policy: sample from distribution with center at x and variance 0.5. we get exploration

## Polynomial basis
See: https://git.isia.fh-salzburg.ac.at/fieldsofwork/tensorflow-for-splineoptimization/code/jupyter-polynomial-bases) multivariate.ipynb.  
We build on this base and modify and extend this code here.  

In [ ]:
from algorithms import multivariate_polynomial_basis

# Test function to approximate
def testf(x, y):
    return 1-x**2-y**2


fig, ax = plt.subplots(subplot_kw=dict(projection='3d'))
fig.suptitle("The function to approximate")

# 20x20 points
xss = np.linspace(-1, 1, 20)
yss = np.linspace(-1, 1, 20)
X, Y = np.meshgrid(xss, yss)
zss = np.array([[testf(x, y) for x in xss] for y in yss])

ax.plot_surface(X, Y, zss, alpha=0.75, rstride=1, cstride=1, color='orangered', edgecolors='k', lw=0.6)
ax.set_xlabel('position')
ax.set_ylabel('velocity')
plt.show()

In [ ]:
deg = 2
basis = multivariate_polynomial_basis.bivar_power_basis(deg)


# Samples (20x20) of testf to approximate
ps = [[x, y] for x in xss for y in yss]
fs = [testf(*p) for p in ps]
zss = np.array([[testf(x, y) for x in xss] for y in yss])

# Approximate
testf_approx = multivariate_polynomial_basis.function_approx(basis, ps, fs)
zss_approx = np.array([[testf_approx(x, y) for x in xss] for y in yss])
err = multivariate_polynomial_basis.l2_difference(testf, testf_approx, ps)

fig, ax = plt.subplots(subplot_kw=dict(projection='3d'))
fig.suptitle("The Approximation")

ax.plot_surface(X, Y, zss_approx, alpha=0.75, rstride=1, cstride=1, color='orangered', edgecolors='k', lw=0.6)
ax.set_title(f"deg={deg}, err={err:.3e}")
ax.set_xlabel('position')
ax.set_ylabel('velocity')
plt.show()

### Convenience measures

In [ ]:
from algorithms import polynomial_agents

# 20x20 points
xss = np.linspace(-1, 1, 20)
yss = np.linspace(-1, 1, 20)
X, Y = np.meshgrid(xss, yss)

ps = [[x, y] for x in xss for y in yss]
fs = [testf(*p) for p in ps]

# Convenience class holding coeffs and methods
p = multivariate_polynomial_basis.MultiVarPoly(dim=2, degree=2)
p.fit(ps, fs)
polynomial_agents.plot(X, Y, p.evaluate(xss,yss), title='Approximation', xlabel='position', ylabel='velocity')

In [ ]:
# The function to approximate was 1-x**2-y**2
# The coefficients are ordered [x^0y^0, x^1y^0, x^2y^0, x^0y^1, x^1y^1
p.coeffs

### Test n-variate generalization

by comparing with bi-variate implementation.

In [ ]:
# model_power_2 = multivariate_polynomial_basis.bivar_power_basis(2)
# coeffs_model_power_2 = multivariate_polynomial_basis.function_approx_coeffs(model_power_2, ps, fs)
# model_power_2_n = multivariate_polynomial_basis.nvar_power_basis(2, 2)
# coeffs_model_power_2_n = multivariate_polynomial_basis.function_approx_coeffs(model_power_2_n, ps, fs)
# model_power_3 = multivariate_polynomial_basis.bivar_power_basis(3)
# coeffs_model_power_3 = multivariate_polynomial_basis.function_approx_coeffs(model_power_3, ps, fs)
# model_power_3_n = multivariate_polynomial_basis.nvar_power_basis(2, 3)
# coeffs_model_power_3_n = multivariate_polynomial_basis.function_approx_coeffs(model_power_3_n, ps, fs)

# model_chebyshev_2 = multivariate_polynomial_basis.bivar_chebyshev_basis(2)
# coeffs_model_chebyshev_2 = multivariate_polynomial_basis.function_approx_coeffs(model_chebyshev_2, ps, fs)
# model_chebyshev_2_n = multivariate_polynomial_basis.nvar_chebyshev_basis(2, 2)
# coeffs_model_chebyshev_2_n = multivariate_polynomial_basis.function_approx_coeffs(model_chebyshev_2_n, ps, fs)
# model_chebyshev_3 = multivariate_polynomial_basis.bivar_chebyshev_basis(3)
# coeffs_model_chebyshev_3 = multivariate_polynomial_basis.function_approx_coeffs(model_chebyshev_3, ps, fs)
# model_chebyshev_3_n = multivariate_polynomial_basis.nvar_chebyshev_basis(2, 3)
# coeffs_model_chebyshev_3_n = multivariate_polynomial_basis.function_approx_coeffs(model_chebyshev_3_n, ps, fs)

# Training: REINFORCE

https://pytorch.org/docs/stable/distributions.html:  
**_REINFORCE is commonly seen as the basis for policy gradient methods in reinforcement learning_**

A classical REINFORCE update at time $t$ involves not all actions, but only the action $a$ taken at time $t$.   
REINFORCE falls under the category of Monte-Carlo Policy-Gradient Control as outlined in S&B p.328, where weights are updated as   
\
$\theta_\mu \gets \theta_\mu + \alpha\gamma^tG\nabla \ln \pi(a|s, \theta_\mu),$  
$\theta_\sigma \gets \theta_\sigma + \alpha\gamma^tG\nabla \ln \pi(a|s, \theta_\sigma).$  
\
with:  
\
$G \gets \sum_{k=t+1}^{T}\gamma^{k-t-1}R_k$,  
\
The term $\alpha\gamma^tG\nabla \ln \pi(a|s, \theta_\sigma)$ is called the **score function**.  
$\pi(a|s, \theta_\sigma)$ is the **normal probability density function** (S&B p.335).  
The respective derivatives of this **log probability** evaluate to  
\
$\nabla \ln \pi(a|s, \theta_\mu) = \frac{1}{\sigma(s)^2}(a-\mu(s)x_\mu(s),$  
$\nabla \ln \pi(a|s, \theta_\sigma) = ( \frac{(a-\mu(s))^2}{\sigma(s)}-1 ) x_\sigma(s).$   
\
Using the notation of S&B, $x_\mu(s)$ and $x_\sigma(s)$ is the evaluation of the basis vectors of our polynomial approximators at location $S$ (without coefficients).  
This means that, if we take the partial derivative with respect to a specific parameter $\nabla\theta_i$, what remains is the $i$-th basis vector of our polynomial approximator (i.e. the $i$-th Chebyshev polynomials).  

### Some tests without parallelization

In [ ]:
degree=3
alpha=0.0003
alpha_mu=0.0003
alpha_sigma=0.00003
episodes=3
discount=0.9 # discount=1.0 diverges for sigma approximator
initial_sigma=0.25 # The higher, the more initial exploration

env = gym.make("MountainCarContinuous-v0", render_mode='human')
reinforce_trainable_mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev',
                                                                                    degree=degree,
                                                                                    initial_sigma=initial_sigma,
                                                                                    normalize_observations=False)
reinforce_rewards = []
reinforce_steps = []
reinforce_coeffs = []
reinforce_loss = []
_, _, _, _ = reinforce_trainable_mrp.train(alpha_mu=alpha_mu, alpha_sigma=alpha_sigma, epochs=episodes, discount=discount,
                                                            method='reinforce_autodiff',
                                                            learning_history=reinforce_rewards,
                                                            steps_history=reinforce_steps,
                                                            coeffs_history=reinforce_coeffs,
                                                            loss_history=reinforce_loss)

## Degree 3: Without normalization of observations

In [ ]:
alpha=0.0003
alpha_mu=0.0003
alpha_sigma=0.00003
episodes=100
discount=0.9 # discount=1.0 diverges for sigma approximator
initial_sigma=0.25 # The higher, the more initial exploration

In [ ]:
fig = plt.figure(figsize=(14, 7))
ax1 = fig.add_subplot(131, projection='3d')
ax2 = fig.add_subplot(132, projection='3d')

env = gym.make("MountainCarContinuous-v0", render_mode="human") 
#env = gym.make("MountainCarContinuous-v0") 
reinforce_trainable_mrp_max_deg3 = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=False, initial_sigma=initial_sigma)
obs = reinforce_trainable_mrp_max_deg3.reset()[0]

# how does this random policy look like?
# 20x20 points
xss = np.linspace(-1, 1, 20)
yss = np.linspace(-1, 1, 20)
X, Y = np.meshgrid(xss, yss)

polynomial_agents.plot(X, Y, reinforce_trainable_mrp_max_deg3.agent.evaluate_mu(xss,yss), title=r'$\mu$ initialization', xlabel='position', ylabel='velocity', zlabel=r'$\mu(s)$', ax=ax1)
polynomial_agents.plot(X, Y, reinforce_trainable_mrp_max_deg3.agent.evaluate_sigma(xss,yss), title=r'$\sigma$ initialization', xlabel='position', ylabel='velocity', zlabel=r'$\sigma(s)$', ax=ax2)

Since randomness is involved and the result heavily depends on whether exploration finds the goal flag early on,  
we run n_runs parallel trainings and choose the best result.  

In [ ]:
mp.cpu_count()

In [ ]:
kwargs = {'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce', 'normalize_observations': False}

n_runs = 5
args = [kwargs for i in range(n_runs)]

In [ ]:
with mp.Pool(processes=n_runs) as pool:
    reinforce_trainable_mrp_max_deg3_results = pool.map(parallel.job_reinforce_train, args)

# Permanently store results
db['reinforce_trainable_mrp_max_deg3_results'] = reinforce_trainable_mrp_max_deg3_results

In [ ]:
# find the maximum of column 0
data = [
    [1, 5, 3],
    [4, 2, 6],
    [7, 8, 9],
    [3, 4, 1]
]

max(row[0] for row in data)

In [ ]:
# row index of maximum
max(enumerate(data), key=lambda x: x[1][0])[0]

In [ ]:
max(enumerate(reinforce_trainable_mrp_max_deg3_results), key=lambda x: x[1][0])[0]

In [ ]:
reinforce_trainable_mrp_max_deg3_results = db['reinforce_trainable_mrp_max_deg3_results']
best_index = max(enumerate(reinforce_trainable_mrp_max_deg3_results), key=lambda x: x[1][0])[0]
reinforce_trainable_mrp_max_deg3 = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=False, initial_sigma=initial_sigma, mu_coeffs=reinforce_trainable_mrp_max_deg3_results[best_index][-1], sigma_coeffs=reinforce_trainable_mrp_max_deg3_results[best_index][-2])
obs = reinforce_trainable_mrp_max_deg3.reset()[0]
reinforce_trainable_mrp_max_deg3.agent.plot_me()

In [ ]:
plt.plot(reinforce_trainable_mrp_max_deg3_results[best_index][1], label='rewards')
plt.legend(loc='best')
plt.title(f'Cumulated reward over episodes, best result out of {n_runs} runs')

In [ ]:
reinforce_trainable_mrp_max_deg3_mean_results = np.mean([m[1] for m in reinforce_trainable_mrp_max_deg3_results], axis=0)
window_size=5
reinforce_trainable_mrp_max_deg3_moving_average_results = np.convolve(reinforce_trainable_mrp_max_deg3_mean_results, np.ones(window_size), mode='valid') / window_size

plt.plot(reinforce_trainable_mrp_max_deg3_mean_results, label='mean rewards')
plt.plot(reinforce_trainable_mrp_max_deg3_moving_average_results, label=f'moving average ({window_size})')
plt.legend(loc='best')
plt.title(f'Cumulated reward over episodes, mean result over {n_runs} runs\nTotal: {np.sum(reinforce_trainable_mrp_max_deg3_mean_results)}')

In [ ]:
# # Run some actions from trained policy
# obs = reinforce_trainable_mrp_max_deg3.reset()[0]
# for i in range(700):
#     reinforce_trainable_mrp_max_deg3.render()
#     obs, reward, terminated, truncated, info, action, _ = reinforce_trainable_mrp_max_deg3.step(obs)  
#     if terminated:
#         s = reinforce_trainable_mrp_max_deg3.reset()

### Interpretation

Although exploration frequently finds the goal flag, the mean reward only increases slowly.  
Running evaluation with the best result shows that the car does not manage to reach the goal flag.  

## Degree 3: Normalization of observations

Gymnasium offers ObservationWrappers: [https://gymnasium.farama.org/api/wrappers/observation_wrappers/](https://gymnasium.farama.org/api/wrappers/observation_wrappers/)

In [ ]:
alpha=0.0003
alpha_mu=0.0003
alpha_sigma=0.00003
episodes=100
discount=0.9 # discount=1.0 diverges for sigma approximator
initial_sigma=0.25 # The higher, the more initial exploration

In [ ]:
env = gym.make("MountainCarContinuous-v0", render_mode="human") 
#env = gym.make("MountainCarContinuous-v0") 
reinforce_trainable_mrp_max_deg3_norm = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=True, initial_sigma=initial_sigma)
obs = reinforce_trainable_mrp_max_deg3_norm.reset()[0]
reinforce_trainable_mrp_max_deg3_norm.agent.plot_me()

In [ ]:
env.observation_space.low

In [ ]:
env.observation_space.high

In [ ]:
reinforce_trainable_mrp_max_deg3_norm.normalize(0.6, env.observation_space.low[0], env.observation_space.high[0])

In [ ]:
reinforce_trainable_mrp_max_deg3_norm.normalize(0.01, env.observation_space.low[1], env.observation_space.high[1])

In [ ]:
reinforce_trainable_mrp_max_deg3_norm.normalize([0.6, 0.07], env.observation_space.low, env.observation_space.high)

In [ ]:
reinforce_trainable_mrp_max_deg3_norm.normalize([-1.2, -0.07], env.observation_space.low, env.observation_space.high)

In [ ]:
reinforce_trainable_mrp_max_deg3_norm.normalize([-0.3, 0], env.observation_space.low, env.observation_space.high)

In [ ]:
kwargs = {'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce', 'normalize_observations': True}

n_runs = 5
args = [kwargs for i in range(n_runs)]

with mp.Pool(processes=n_runs) as pool:
    reinforce_trainable_mrp_max_deg3_norm_results = pool.map(parallel.job_reinforce_train, args)

# Permanently store results
db['reinforce_trainable_mrp_max_deg3_norm_results'] = reinforce_trainable_mrp_max_deg3_norm_results

In [ ]:
reinforce_trainable_mrp_max_deg3_norm_results = db['reinforce_trainable_mrp_max_deg3_norm_results']

best_index_reinforce = max(enumerate(reinforce_trainable_mrp_max_deg3_norm_results), key=lambda x: x[1][0])[0]
reinforce_trainable_mrp_max_deg3_norm = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=True, initial_sigma=initial_sigma, mu_coeffs=reinforce_trainable_mrp_max_deg3_norm_results[best_index_reinforce][-1], sigma_coeffs=reinforce_trainable_mrp_max_deg3_norm_results[best_index_reinforce][-2])
obs = reinforce_trainable_mrp_max_deg3_norm.reset()[0]
reinforce_trainable_mrp_max_deg3_norm.agent.plot_me()

In [ ]:
plt.plot(reinforce_trainable_mrp_max_deg3_norm_results[best_index][1], label='rewards')
plt.legend(loc='best')
plt.title(f'Cumulated reward over episodes, best result out of {n_runs} runs')

In [ ]:
reinforce_trainable_mrp_max_deg3_norm_mean_results = np.mean([m[1] for m in reinforce_trainable_mrp_max_deg3_norm_results], axis=0)
window_size=5
reinforce_trainable_mrp_max_deg3_norm_moving_average_results = np.convolve(reinforce_trainable_mrp_max_deg3_norm_mean_results, np.ones(window_size), mode='valid') / window_size

plt.plot(reinforce_trainable_mrp_max_deg3_norm_mean_results, label='mean rewards')
plt.plot(reinforce_trainable_mrp_max_deg3_norm_moving_average_results, label=f'moving average ({window_size})')
plt.legend(loc='best')
plt.title(f'Cumulated reward over episodes, mean result over {n_runs} runs\nTotal: {np.sum(reinforce_trainable_mrp_max_deg3_norm_mean_results)}')

In [ ]:
# # Run some actions from trained policy
# obs = reinforce_trainable_mrp_max_deg3_norm.reset()[0]
# for i in range(700):
#     reinforce_trainable_mrp_max_deg3_norm.render()
#     obs, reward, terminated, truncated, info, action, _ = reinforce_trainable_mrp_max_deg3_norm.step(obs)  
#     if terminated:
#         s = reinforce_trainable_mrp_max_deg3_norm.reset()

### Interpretation

The mean reward inreases significantly while training progresses.  The result is superior to the one without normalization.  
Running evaluation with the best result shows that the car reaches the goal flag. Success!

## REINFORCE Utilizing Autodiff: Adam Optimizer

Instead of computing $\nabla \ln \pi(a|s, \theta_\mu)$ and updating weights manually, we can utilize Autodiff functionality of PyTorch.    
Utilizing Autodiff and corresponding optimizers requires the minimization of loss functions that the framework can differentiate to update the policy parameters.  
We can minimize the negative of our REINFORCE score function objective to use typical gradient descent optimization routines as   
    
\
$\ell = -\sum_t\gamma^tG_t\ln \pi(a|s, \theta)$,
\
with
\
$G_t \gets \sum_{k=t+1}^{T}\gamma^{k-t-1}R_k$
\
as usual.

According to S&B p.335, for continuous action spaces, $\pi(a|s, \theta)$ is the *probability density*:  
\
$\pi(a|s) = \frac{1}{\sigma(s)\sqrt{2\pi}}\mathbb{e}^{-\frac{(a-\mu(s))^2}{2\sigma(s)^2}}$

In [ ]:
alpha=0.0003
alpha_mu=0.0003
alpha_sigma=0.00003
episodes=100
discount=0.9 # discount=1.0 diverges for sigma approximator
initial_sigma=0.25 # The higher, the more initial exploration

In [ ]:
env = gym.make("MountainCarContinuous-v0", render_mode="human") 
#env = gym.make("MountainCarContinuous-v0") 
reinforce_trainable_mrp_max_deg3_autodiff = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=True, initial_sigma=initial_sigma)
obs = reinforce_trainable_mrp_max_deg3_autodiff.reset()[0]
reinforce_trainable_mrp_max_deg3_autodiff.agent.plot_me()

In [ ]:
kwargs = {'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce_autodiff', 'normalize_observations': True}

n_runs = 5
args = [kwargs for i in range(n_runs)]

with mp.Pool(processes=n_runs) as pool:
    reinforce_trainable_mrp_max_deg3_autodiff_results = pool.map(parallel.job_reinforce_train, args)

# Permanently store results
db['reinforce_trainable_mrp_max_deg3_autodiff_results'] = reinforce_trainable_mrp_max_deg3_autodiff_results

In [ ]:
reinforce_trainable_mrp_max_deg3_autodiff_results = db['reinforce_trainable_mrp_max_deg3_autodiff_results']
best_index_autodiff_results = max(enumerate(reinforce_trainable_mrp_max_deg3_autodiff_results), key=lambda x: x[1][0])[0]
reinforce_trainable_mrp_max_deg3_autodiff = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=True, initial_sigma=initial_sigma, mu_coeffs=reinforce_trainable_mrp_max_deg3_autodiff_results[best_index_autodiff_results][-1], sigma_coeffs=reinforce_trainable_mrp_max_deg3_autodiff_results[best_index_autodiff_results][-2])
obs = reinforce_trainable_mrp_max_deg3_autodiff.reset()[0]
reinforce_trainable_mrp_max_deg3_autodiff.agent.plot_me()

In [ ]:
plt.plot(reinforce_trainable_mrp_max_deg3_autodiff_results[best_index_autodiff_results][1], label='rewards')
plt.legend(loc='best')
plt.title(f'Adam: Cumulated reward over episodes, best result out of {n_runs} runs')

In [ ]:
reinforce_trainable_mrp_max_deg3_autodiff_mean_results = np.mean([m[1] for m in reinforce_trainable_mrp_max_deg3_autodiff_results], axis=0)
window_size=5
reinforce_trainable_mrp_max_deg3_autodiff_moving_average_results = np.convolve(reinforce_trainable_mrp_max_deg3_autodiff_mean_results, np.ones(window_size), mode='valid') / window_size

plt.plot(reinforce_trainable_mrp_max_deg3_autodiff_mean_results, label='mean rewards')
plt.plot(reinforce_trainable_mrp_max_deg3_autodiff_moving_average_results, label=f'moving average ({window_size})')
plt.legend(loc='best')
plt.title(f'Adam: Cumulated reward over episodes, mean result over {n_runs} runs\nTotal: {np.sum(reinforce_trainable_mrp_max_deg3_autodiff_mean_results)}')

### Interpretation

In [ ]:
# # Run some actions from trained policy
# obs = reinforce_trainable_mrp_max_deg3_autodiff.reset()[0]
# for i in range(1000):
#     reinforce_trainable_mrp_max_deg3_autodiff.render()
#     obs, reward, terminated, truncated, info, action, _ = reinforce_trainable_mrp_max_deg3_autodiff.step(obs)  
#     if terminated:
#         s = reinforce_trainable_mrp_max_deg3_autodiff.reset()

### Interpretation

Up to around episode $80$ training the mean reward increases significantly during training, surpassing the normalized REINFORCE update.  
While the moving average reward with the normalized REINFORCE update reaches values of up to $-60$, with the Adam optimizer we reach values close to $-40$.  
Evaluating the best result shows that the car manages to reach the goal flag. Success!  
It is interesting though that we see a decline and subsequent increase of the mean reward after 80 episodes.
We see a milder decline around that episode count also with the standard REINFORCE update.  
Exploration seems to follow another possible path and continued training or early stopping provides better end results.  

## Training and evaluation with available optimizers

### REINFORCE Training

20 Policies trained for 100 episodes.

Some tests

In [ ]:
# find the maximum of column 0
data = [
    [1, 5, 3],
    [4, 2, 6],
    [7, 8, 9],
    [3, 4, 1]
]

max(row[0] for row in data)

In [ ]:
# mean of column
np.mean(np.array(data), axis=0)

Got it - lets continue

In [ ]:
alpha=0.0003
alpha_mu=0.0003
alpha_sigma=0.00003
episodes=100
discount=0.9 # discount=1.0 diverges for sigma approximator
initial_sigma=0.25 # The higher, the more initial exploration
n_runs= 20

optimizers = ['adam', 'adam-amsgrad', 'adamw', 'adamw-amsgrad', 'adamax', 'lbfgs', 'nadam', 'radam', 'rmsprop', 'rprop', 'sgd-momentum', 'sgd-nesterov']
kwargs = {'mode': 'optimizers', 'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce_autodiff', 'normalize_observations': True, 'n_runs': n_runs}

In [ ]:
pool = parallel.NestablePool(n_runs)
reinforce_results_train_optimizers = pool.starmap(parallel.job_reinforce_optimizers, zip(optimizers, repeat(kwargs)))
# Storing variables
db['reinforce_results_train_optimizers'] = reinforce_results_train_optimizers

In [ ]:
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']

In [ ]:
for i in range(20):
    print(f'{np.sum(reinforce_results_train_optimizers[2][i][3])}')

### Plot Training Rewards

In [ ]:
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']

fig, axes = plt.subplots(4, (len(optimizers)+2)//4) 
axes = axes.flatten()
fig.set_figwidth(len(optimizers)*3)
fig.set_figheight(20)
fig.suptitle(f'Min, mean and max rewards over episodes with different optimizers, {episodes} episodes, {n_runs} runs')

plot_training_min_mean_max_rewards(reinforce_results_train_optimizers, optimizers, mcolors.CSS4_COLORS['steelblue'], axes)

### Interpretation

Insights:  
 - lbfgs, rprop, sgd-momentum and sgd-nesterov do not converge  
 - adam and nadam are in a (temporary?) downward tendency when training stops. early stopping or continued training could mitigate this effect.
 - Interestingly, performance of amsgrad versions of optimizers stays behind.    
It seems that the "tamer" behaviour that is beneficial for convergence of supervised learning tasks is not beneficial in this more complex,"exploration heavy", scenario.   

### Policy "shapes"

In [ ]:
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']
best_index, _ = get_best_candidates_of_training_result(reinforce_results_train_optimizers, optimizers, n_runs)

fig, axes = plt.subplots(4, (len(optimizers)+2)//4, subplot_kw={'projection': '3d'}) 
axes = axes.flatten()
fig.set_figwidth(len(optimizers)*3)
fig.set_figheight(22)
#fig.suptitle(f'Approximator plots for best result out of {n_runs} runs')

for i, opt in enumerate(optimizers):
    ax = axes[i]
    try:
        mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=True, initial_sigma=initial_sigma, mu_coeffs=reinforce_results_train_optimizers[i][best_index[i]][-1], sigma_coeffs=reinforce_results_train_optimizers[i][best_index[i]][-2])
        p = mrp.agent.plot_me(ax=ax)
    except:
        pass
    ax.set_title("%s" % optimizers[i])

plt.tight_layout()
plt.show()

In [ ]:
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']
best_index, _ = get_best_candidates_of_training_result(reinforce_results_train_optimizers, optimizers, n_runs)

fig, axes = plt.subplots(4, (len(optimizers)+2)//4) 
axes = axes.flatten()
fig.set_figwidth(len(optimizers)*3)
fig.set_figheight(22)
#fig.suptitle(f'Approximator plots for best result out of {n_runs} runs')

for i, opt in enumerate(optimizers):
    ax = axes[i]
    try:
        mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=True, initial_sigma=initial_sigma, mu_coeffs=reinforce_results_train_optimizers[i][best_index[i]][-1], sigma_coeffs=reinforce_results_train_optimizers[i][best_index[i]][-2])
        p = mrp.agent.plot_me(ax=ax, heatmap=True)
    except:
        pass
    ax.set_title("%s" % optimizers[i])

plt.tight_layout()
plt.show()

## Evaluation / Choosing candidate policy

In [ ]:
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']
#reinforce_trainable_mrp_max_deg3_norm_results = db['reinforce_trainable_mrp_max_deg3_norm_results']
optimizers = ['adam', 'adam-amsgrad', 'adamw', 'adamw-amsgrad', 'adamax', 'lbfgs', 'nadam', 'radam', 'rmsprop', 'rprop', 'sgd-momentum', 'sgd-nesterov']

n_runs = len(reinforce_results_train_optimizers[0])
coeffs, opt_indices = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers, optimizers, n_runs)
# add vanilla REINFORCE update result from earlier
# coeffs.append([row[-1] for row in reinforce_trainable_mrp_max_deg3_norm_results])
# opt_indices.append('REINFORCE')

In [ ]:
episodes = 50
kwargs = {'episodes': episodes, 'return_coeffs': True}

In [ ]:
pool = parallel.NestablePool(mp.cpu_count())
reinforce_results_eval_optimizers = pool.starmap(parallel.job_evaluate_ncoeffs, zip(coeffs, repeat(kwargs)))
# Storing variables
#db['reinforce_results_eval_optimizers'] = reinforce_results_eval_optimizers
db['reinforce_results_eval_optimizers_'] = reinforce_results_eval_optimizers

In [ ]:
reinforce_results_eval_optimizers = db['reinforce_results_eval_optimizers_']
optimizers = ['adam', 'adam-amsgrad', 'adamw', 'adamw-amsgrad', 'adamax', 'lbfgs', 'nadam', 'radam', 'rmsprop', 'rprop', 'sgd-momentum', 'sgd-nesterov']
coeffs, opt_indices = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers, optimizers, n_runs)
# add vanilla REINFORCE update result from earlier
# coeffs.append([row[-1] for row in reinforce_trainable_mrp_max_deg3_norm_results])
# opt_indices.append('REINFORCE')

fig, axes = plt.subplots(4, 3) 
axes = axes.flatten()
fig.set_figwidth(len(opt_indices)*3)
fig.set_figheight(20)
fig.suptitle(f'Evaluation: Min, Mean and Max rewards over episodes with different optimizers, {episodes} episodes per policy')

plot_evaluation_min_mean_max_rewards(reinforce_results_eval_optimizers, opt_indices, window_size=5, color=mcolors.CSS4_COLORS['steelblue'], axes=axes)

In [ ]:
def get_all_eval_results_per_optimizer(results, optimizers):
    ret = {}
    for i, opt in enumerate(optimizers):
        temp_list = []
        for j in range(len(results[i])):
            try:
                temp_list.append(results[i][j][1])
            except:
                pass
        ret[opt] = np.concatenate(temp_list)
    return ret

In [ ]:
optimizers = ['Adam', 'Adam-AMSGrad', 'AdamW', 'AdamW-AMSGrad', 'Adamax', 'lbfgs', 'NAdam', 'RAdam', 'RMSprop', 'Rprop', 'sgd-momentum', 'sgd-nesterov']
n_runs = 20
episodes = 50
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']
coeffs, opts = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers, optimizers, n_runs)

In [ ]:
axes_indices = {'Adam': 0, 'Adam-AMSGrad': 1, 'AdamW': 2, 'AdamW-AMSGrad': 3, 'Adamax': 4, 'NAdam': 5, 'RAdam': 6, 'RMSprop': 7, 'Rprop': 8}

reinforce_results_eval_optimizers = db['reinforce_results_eval_optimizers_']

results_chebyshev = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers, opts)

dicts = [results_chebyshev]
all_results = {}

for d in dicts:
    for key, value in d.items():
        if key in all_results:
            all_results[key].append(value)
        else:
            all_results[key] = [value]  

fig, axes = plt.subplots(3, 3)
axes = axes.flatten()
fig.set_figwidth(24)
fig.set_figheight(18)
fig.suptitle(f'Evaluation rewards with different optimizers, number of converging policies utilized is shown in brackets \n {episodes} episodes per policy, each datapoint is the cumulated reward of one episode')

for i, opt in enumerate(all_results):
    # if opt == 'Rprop':
    #     break
    ax = axes[axes_indices[opt]]
    optim_boxplotdata = []
    num_policies = []
    for j, _ in enumerate(all_results[opt]):
        try:
            optim_boxplotdata.append(all_results[opt][j])
        except:
            pass
        num_policies.append(len(all_results[opt][j])/episodes)
    ax.set_title(opt + f' ({num_policies[0]:.0f})', fontsize=24)
    ax.boxplot(optim_boxplotdata)
    #ax.set_xticks([1], [f'Ch-3'])
    ax.xaxis.set_visible(False)
    ax.set_ylim([-200, 110])
    ax.tick_params(axis='both', labelsize=20)

    #fig.tight_layout()
    fig.savefig("ch3_training_evaluation.pdf")

### Interpretation

Insights:  
 - Optimizers adam, adamw, nadam, radam and rmsprop managed to converge to reaching the goal flag in at least one instance

### Analyze best result

In [ ]:
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']
reinforce_results_eval_optimizers = db['reinforce_results_eval_optimizers_']
n_runs = 20
optimizers = ['adam', 'adam-amsgrad', 'adamw', 'adamw-amsgrad', 'adamax', 'lbfgs', 'nadam', 'radam', 'rmsprop', 'rprop', 'sgd-momentum', 'sgd-nesterov']
coeffs, opt_indices = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers, optimizers, n_runs)
# # Attention: coeffs is missing non-converging entries, so reinforce_results_train_optimizers[opt][best_index[opt]][-1] != coeffs[opt]
best_opt_index, best_opt_run = get_best_candidates_of_evaluation_result(reinforce_results_eval_optimizers)
best_opt_index, best_opt_run 

In [ ]:
db['chebyshev_reinforce_best_single_episode_result_coeffs'] = reinforce_results_eval_optimizers[2][7][0]
reinforce_results_eval_optimizers[2][7][0]

In [ ]:
kwargs = {'start_loc': -0.6, 'degree': 3}

results = []

for c in coeffs:
    pool = mp.Pool(mp.cpu_count())
    results.append(pool.starmap(parallel.job_get_episode_reward, zip(c, repeat(kwargs))))

db['reinforce_results_single_episode'] = results

In [ ]:
env = gym.make("MountainCarContinuous-v0", render_mode="human")
mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3,
                                                               normalize_observations=True)
results = db['reinforce_results_single_episode']
max_reward, optimizer, run, s, v = get_best_single_episode_evaluation_result(results, mrp)

In [ ]:
start=-0.6
coeffs = db['chebyshev_reinforce_best_single_episode_result_coeffs']

# Run some actions from trained policy
env = gym.make("MountainCarContinuous-v0", render_mode="human")

mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3,
                                                               normalize_observations=True,
                                                               mu_coeffs=db['chebyshev_reinforce_best_single_episode_result_coeffs'])

r = 0.0
observations = []
obs = mrp.reset(options={'low': start, 'high': start})[0]
observations.append(obs)
print(f'Starting at {mrp.unnormalize(observations[0], np.array([0.6, 0.07]), np.array([-1.2, -0.07]))}')
for i in range(1000):
    obs, reward, terminated, truncated, info, action, _ = mrp.step(obs)
    r += reward
    observations.append(obs)
    #print(reward, r)
    if terminated or truncated:
        break
r, i, mrp.unnormalize(obs, np.array([0.6, 0.07]), np.array([-1.2, -0.07]))

In [ ]:
db["chebyshev_episode_reward"] = max_reward
db["chebyshev_episode_len"] = s
db["chebyshev_v_target"] = v

In [ ]:
#fig, (ax1, ax2) = plt.subplots(1,2) 
fig, ax1 = plt.subplots() 
fig.set_figwidth(10)
fig.set_figheight(8)

env = gym.make("MountainCarContinuous-v0", render_mode="human")
mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=True, mu_coeffs=db['chebyshev_reinforce_best_single_episode_result_coeffs'])
polynomial_agents.plot_heatmap(mrp, fig=fig, ax=ax1, title='Policy Heatmap: Chebyshev Max.Degree 3\n (Unnormalized)', unnormalize=True)
#polynomial_agents.plot_heatmap(mrp, fig=fig, ax=ax2, title='Policy Heatmap: Chebyshev Max.Degree 3\n (Normalized)', unnormalize=False)

In [ ]:
# https://gist.github.com/botforge/64cbb71780e6208172bbf03cd9293553
from matplotlib import animation


def save_frames_as_gif(frames, path='./', filename='gym_animation.gif'):

    #Mess with this to change frame size
    plt.figure(figsize=(frames[0].shape[1] / 72.0, frames[0].shape[0] / 72.0), dpi=72)

    patch = plt.imshow(frames[0])
    plt.axis('off')

    def animate(i):
        patch.set_data(frames[i])

    anim = animation.FuncAnimation(plt.gcf(), animate, frames = len(frames), interval=50)
    anim.save(path + filename, writer='imagemagick', fps=60)


def run_chebyshev_model(coeffs, episodes=1, env_name="MountainCarContinuous-v0", degree=3, normalize_observations=True):
    env = gym.make(env_name, render_mode='rgb_array')
    mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=degree,
                                                                    normalize_observations=normalize_observations,
                                                                    initial_sigma=0.25,
                                                                    mu_coeffs=coeffs)  

    frames = []

    for e in range(episodes):
        obs = mrp.reset()[0]

        print(".", end="")

        episode_reward = 0.0
        observations = []
        episode_len = 0

        
        # Run actions from trained policy
        while True:
            frames.append(env.render()) # Workaround for render mode "human" crashing kernel: https://github.com/openai/gym/issues/3031
                # plt.imshow(img)
                # display.clear_output(wait=True)
                # display.display(plt.gcf())
            observations.append(obs)
            obs, reward, terminated, truncated, info, action, _ = mrp.step(obs)
            episode_reward += reward
            episode_len += 1
            if terminated or truncated:
                print(f"Episode Reward: {episode_reward:.2f} after {episode_len} steps")
                break

        print("#", end="")

    env.close()
    save_frames_as_gif(frames)

    return episode_reward, observations

In [ ]:
run_chebyshev_model(coeffs=db['chebyshev_reinforce_best_single_episode_result_coeffs'], episodes=3)

In [ ]:
#fig, (ax1, ax2) = plt.subplots(1,2) 
fig, ax1 = plt.subplots() 
fig.set_figwidth(10)
fig.set_figheight(8)

env = gym.make("MountainCarContinuous-v0", render_mode="human")
mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='chebyshev', degree=3, normalize_observations=True, mu_coeffs=db['chebyshev_reinforce_best_single_episode_result_coeffs'])
#polynomial_agents.plot_heatmap(mrp, fig=fig, ax=ax1, title='Policy Heatmap: Chebyshev Max.Degree 3\n (Unnormalized)', unnormalize=True, trajectory=observations, trajectory_label=f'Trajectory starting at x0 = {start}\nReward: {r:.4f}')
#polynomial_agents.plot_heatmap(mrp, fig=fig, ax=ax2, title='Policy Heatmap: Chebyshev Max.Degree 3\n (Normalized)', unnormalize=False)
polynomial_agents.plot_heatmap(mrp, fig=fig, ax=ax1, unnormalize=True, trajectory=observations, trajectory_label=f'R = {r:.4f}', actionbar=False)
fig.savefig(".\\paper-figures\\policyheatmap_chebyshev3_trajectory_x0.pdf")